# Clase 2 — Las particularidades del dato espacial

**Sistemas de Información Geográfica**
Especialización en Ciencias Sociales Computacionales — Universidad Nacional Guillermo Brown

| | |
|---|---|
| **Unidad del programa** | 2 — Particularidades de la información geográfica |
| **Duración** | 3 horas (≈ 35 min de presentación + 130 min de notebook + 15 min de cierre) |
| **Versión** | 2026.1 |
| **Docente** | Renzo Polo |
| **Licencia** | CC BY-SA 4.0 |

---

## 1. La pregunta de hoy

> ### ¿Cambia la respuesta si cambio la grilla?

En la clase pasada calculamos escuelas por habitante **por provincia**, y descubrimos que
el conteo absoluto y la tasa daban mapas opuestos. Hoy vamos más al fondo: vamos a ver que
**la propia elección de la unidad territorial modifica el resultado**, y que esa elección
suele tomarse por conveniencia —"tenía los datos por departamento"— sin que nadie la
justifique.

Esto no es una curiosidad técnica. Si el máximo de un indicador cambia según dónde
dibujemos las líneas de una grilla, entonces frases como *"la zona más crítica del
aglomerado"* dependen de una decisión metodológica que el mapa no muestra.

Los datos espaciales tienen un conjunto de propiedades que los vuelven distintos de una
tabla común, y que **rompen supuestos** de la estadística que solemos dar por sentados.
Hoy las recorremos una por una, midiéndolas.

## 2. Objetivos de esta clase

Al terminar, deberías poder:

1. **Explicar** la primera ley de la geografía y por qué amenaza el supuesto de
   independencia de las observaciones.
2. **Demostrar** que la longitud de un objeto geográfico depende de la escala a la que se lo mida.
3. **Distinguir** los dos componentes del problema de la unidad de área modificable:
   el efecto de agregación y el de zonificación.
4. **Cuantificar** el efecto de borde en una zona de estudio.
5. **Reconocer** cuándo un valor agregado por área representa mal a la población que contiene.
6. **Identificar** una falacia ecológica en un razonamiento.

### Lo que esta clase NO cubre

- **Cómo medir la autocorrelación espacial.** Hoy la vemos de manera intuitiva y sin
  fórmulas. El índice de Moran y los métodos formales son la **Clase 7**.
- **Cómo elegir el CRS correcto.** Hoy usamos proyecciones adecuadas sin explicarlas del
  todo; el tema es la **Clase 3**.
- **Cómo corregir estos problemas.** Hoy los diagnosticamos. Algunas correcciones aparecen
  en las clases 7 y 8; otras no tienen solución técnica y solo se declaran.

## 3. Prerrequisitos

- Clase 1: cargar capas, inspeccionarlas, unir tablas.
- `pandas`: `groupby()`, `merge()`, correlaciones.

## 3b. Material de esta clase

📽️ **Presentación Clase 2**, diapositivas 1–16. *(Las diapositivas de formatos de archivo
se vieron en la Clase 1.)*

| Bloque de la notebook | Diapositivas |
|---|---|
| Primera ley de la geografía y autocorrelación | 12 |
| Escala de análisis | 13 |
| Unidad de área modificable | 14 |
| Efecto de borde | 15 |
| Localización representada | 16 |

📖 Olaya, V. *Sistemas de Información Geográfica*, capítulo 9.

## 4. Preparación del entorno

In [ ]:
!pip install -q "geopandas==1.0.1" "mapclassify==2.8.1" "folium==0.17.0" "matplotlib==3.9.2"
!wget -q -O sig_utils.py https://raw.githubusercontent.com/renzoepolo/sig-ciencias-sociales/main/sig_utils.py

import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
from sig_utils import cargar, grilla, chequear_crs, CRS_ARGENTINA

# Sistemas de referencia que vamos a usar hoy.
# La Clase 3 explica por qué estos y no otros; por ahora, alcanza con saber que
# están en metros y que respetan superficies y distancias en Argentina.
EQUIVALENTE  = CRS_ARGENTINA["sudamerica_equivalente"]   # para áreas
EQUIDISTANTE = CRS_ARGENTINA["sudamerica_equidistante"]  # para distancias

print(f"GeoPandas {gpd.__version__} — entorno listo")

In [ ]:
provincias = cargar("provincias")
escuelas   = cargar("escuelas")
ruta40     = cargar("ruta40")

---

## 5. Recapitulación: dónde quedamos

En la Clase 1 construimos un indicador de escuelas por cada 10.000 habitantes a nivel
provincial y vimos que el conteo absoluto y la tasa producían mapas opuestos.

Reconstruimos ese indicador en una celda, porque lo vamos a usar todo el día.

In [ ]:
tasa_provincial = (
    provincias
    .merge(escuelas.groupby("provincia").size().reset_index(name="escuelas"),
           on="provincia", how="left")
)
tasa_provincial["tasa"] = (
    tasa_provincial["escuelas"] / tasa_provincial["poblacion"] * 10_000
)

print(f"{len(tasa_provincial)} provincias | "
      f"tasa de {tasa_provincial.tasa.min():.1f} a {tasa_provincial.tasa.max():.1f} "
      f"escuelas cada 10.000 habitantes")

---

## 6. La primera ley de la geografía

📽️ *Presentación Clase 2, diapositiva 12.*

### 🧭 Concepto

Waldo Tobler la enunció en 1970, y es engañosamente simple:

> *"Todo está relacionado con todo lo demás, pero las cosas cercanas están más
> relacionadas entre sí que las lejanas."*

Suena a obviedad. No lo es: si fuera cierta, entonces **dos observaciones vecinas no son
dos observaciones independientes**. Y la independencia de las observaciones es un supuesto
de casi toda la estadística que usamos: de la regresión lineal, de los tests de
significancia, de los intervalos de confianza.

Dicho de otro modo: si los datos espaciales cumplen la ley de Tobler —y en general la
cumplen— entonces al aplicarles estadística convencional estamos **contando la misma
información más de una vez**.

Vamos a comprobar si se cumple, sin ninguna fórmula: comparando cuánto se diferencian las
provincias **vecinas** contra cuánto se diferencian provincias **tomadas al azar**.

In [ ]:
tp = tasa_provincial.to_crs(EQUIVALENTE)

# Diferencia de tasa entre cada par de provincias que comparten frontera
diferencias_vecinas = []
for _, fila in tp.iterrows():
    vecinas = tp[tp.geometry.touches(fila.geometry)]
    for _, vecina in vecinas.iterrows():
        diferencias_vecinas.append(abs(fila.tasa - vecina.tasa))

# Diferencia de tasa entre pares tomados al azar
generador = np.random.default_rng(0)
diferencias_azar = [
    abs(tp.tasa.iloc[a] - tp.tasa.iloc[b])
    for a, b in generador.choice(len(tp), size=(2000, 2))
    if a != b
]

print(f"Pares de provincias vecinas: {len(diferencias_vecinas)}")
print(f"Pares al azar:               {len(diferencias_azar)}")

### ✅ Comprobación

In [ ]:
media_vecinas = np.mean(diferencias_vecinas)
media_azar    = np.mean(diferencias_azar)

print(f"Diferencia media entre provincias VECINAS: {media_vecinas:.2f}")
print(f"Diferencia media entre provincias AL AZAR: {media_azar:.2f}")
print(f"\nLas vecinas se parecen un {100*(1-media_vecinas/media_azar):.0f}% más de lo esperable por azar")

assert media_vecinas < media_azar, "Si esto falla, no hay autocorrelación positiva"

### 🔍 Interpretación

Las provincias vecinas se parecen **un 38 % más** entre sí que dos provincias cualesquiera.
La ley de Tobler se cumple en estos datos.

Eso tiene una consecuencia incómoda: si mañana ajustáramos una regresión con estas 24
provincias como observaciones independientes, estaríamos exagerando cuánta información
tenemos. No tenemos 24 datos independientes; tenemos algo más parecido a 12 o 15.
Los valores p saldrían más chicos de lo que corresponde y concluiríamos que hay efectos
significativos donde tal vez no los haya.

> ⚠️ **Ojo con la interpretación.** Que las provincias vecinas se parezcan **no** significa
> que una influya sobre la otra. Puede ser que compartan clima, historia productiva o
> composición demográfica. La autocorrelación describe un patrón; no identifica su causa.
> Vamos a medirla formalmente en la Clase 7, y ahí insistiremos con esto.

---

## 7. Escala: ¿cuánto mide la Ruta 40?

📽️ *Presentación Clase 2, diapositiva 13.*

### 🧭 Concepto

Parece una pregunta con una sola respuesta. No la tiene.

Cuando representamos un objeto sinuoso —una ruta, una costa, un río— lo hacemos con una
secuencia de vértices. Cuantos más vértices, más curvas capturamos y **más largo resulta**.
Si medimos con menos detalle, las curvas pequeñas desaparecen y el objeto se acorta.

Esto se conoce como la **paradoja de la costa**: la longitud de una costa depende de la
resolución con que se la mide, y no converge a un valor único. La escala no es un detalle
de presentación del mapa; **es parte de la definición de la medición**.

`simplify()` aplica el algoritmo de Douglas-Peucker: elimina los vértices que se apartan
menos que una tolerancia dada de la línea que los aproxima.

### ▶️ Ejecución

Ojo con un detalle que viene de la Clase 1: para medir longitudes hay que estar en un
sistema de coordenadas **en metros**. Si midiéramos en grados, el número no significaría
nada.

In [ ]:
# Reproyectamos a un sistema que conserva distancias en Argentina
ruta40_m = ruta40.to_crs(EQUIDISTANTE)
chequear_crs(ruta40_m, proyectado=True, nombre="Ruta 40")

In [ ]:
def describir(geometria):
    """Cuenta los vértices de una geometría de líneas."""
    return sum(len(parte.coords) for parte in geometria.geoms)

longitud_original = ruta40_m.length.iloc[0] / 1000
vertices_original = describir(ruta40_m.geometry.iloc[0])

filas = [("Detalle completo", longitud_original, vertices_original, 0.0)]
for tolerancia_km in [0.5, 2, 10, 50]:
    simplificada = ruta40_m.geometry.simplify(tolerancia_km * 1000)
    longitud = simplificada.length.iloc[0] / 1000
    filas.append((
        f"Tolerancia {tolerancia_km} km",
        longitud,
        describir(simplificada.iloc[0]),
        100 * (1 - longitud / longitud_original),
    ))

tabla_escala = pd.DataFrame(
    filas, columns=["Nivel de detalle", "Longitud (km)", "Vértices", "Pérdida (%)"]
)
tabla_escala.round(1)

### ✅ Comprobación — ¿el número tiene sentido?

In [ ]:
print(f"Longitud medida con detalle completo: {longitud_original:,.0f} km")
print(f"Longitud oficial declarada por Vialidad Nacional: ~5.194 km")
print(f"Diferencia: {abs(longitud_original - 5194):,.0f} km "
      f"({100*abs(longitud_original-5194)/5194:.1f}%)")

In [ ]:
fig, ejes = plt.subplots(1, 3, figsize=(13, 7))
for eje, tolerancia in zip(ejes, [0, 10, 50]):
    if tolerancia == 0:
        capa, titulo = ruta40_m.geometry, "Detalle completo"
    else:
        capa = ruta40_m.geometry.simplify(tolerancia * 1000)
        titulo = f"Tolerancia {tolerancia} km"
    capa.plot(ax=eje, color="#d95f02", linewidth=1)
    eje.set_title(f"{titulo}\n{capa.length.iloc[0]/1000:,.0f} km · {describir(capa.iloc[0])} vértices")
    eje.set_axis_off()
plt.tight_layout()
plt.show()

### 🔍 Interpretación

La Ruta 40 mide **5.166 km** con el máximo detalle disponible y **4.152 km** generalizada
a 50 km de tolerancia: un 19,6 % menos. Los tres mapas se ven casi iguales a esta escala
de impresión, pero responden distinto a la pregunta "¿cuánto mide?".

Fijate además que los primeros 41.000 vértices que eliminamos cuestan solo un 3,4 % de
longitud: la mayor parte del detalle no aporta información sustantiva, pero sí peso al
archivo.

> ⚠️ **Para tu investigación:** cuando compares magnitudes espaciales entre fuentes
> —longitud de red vial entre provincias, superficie de áreas protegidas entre países—
> asegurate de que provengan de cartografías de **escala comparable**. Una provincia no
> tiene más rutas que otra por estar mejor mapeada; pero el dato puede decir eso.

---

## 8. El problema de la unidad de área modificable (MAUP)

📽️ *Presentación Clase 2, diapositiva 14.*

### 🧭 Concepto

Muchas variables sociales **no se pueden medir en un punto**. La densidad de población, la
tasa de desempleo, el porcentaje de hogares con NBI: todas necesitan un área para existir.
Y las áreas que usamos —provincias, departamentos, radios censales, celdas de una grilla—
son **construcciones administrativas o analíticas**, no rasgos del territorio.

El problema de la unidad de área modificable, formulado por Openshaw en 1984, dice que
**los resultados dependen de esas áreas**. Y tiene dos componentes que conviene separar,
porque se corrigen de manera distinta:

| Componente | Qué cambia | Ejemplo |
|---|---|---|
| **Agregación** (o escala) | El **tamaño** de las unidades | Provincia vs. departamento vs. radio censal |
| **Zonificación** | La **forma o posición**, con el mismo tamaño | Dos grillas del mismo paso, corridas media celda |

> ⚠️ **Un problema del material del año pasado.** Comparábamos una grilla de cuadrados de
> ~10.100 km² contra una de hexágonos de ~8.700 km². Al tener áreas distintas, esa
> comparación mezclaba los dos efectos y no permitía atribuir la diferencia a ninguno.
> Hoy los separamos: cambiamos **una cosa por vez**.

Trabajamos sobre **Mendoza**, que tiene una estructura de poblamiento clara —un oasis
denso y un desierto vacío— donde el efecto se ve bien.

In [ ]:
mendoza          = provincias[provincias.provincia == "Mendoza"]
escuelas_mendoza = escuelas[escuelas.provincia == "Mendoza"]

superficie = mendoza.to_crs(EQUIVALENTE).area.iloc[0] / 1e6
print(f"Mendoza: {len(escuelas_mendoza)} escuelas en {superficie:,.0f} km²")
print(f"Densidad media: {len(escuelas_mendoza)/superficie*100:.2f} escuelas cada 100 km²")

### ▶️ Efecto de zonificación

Tres grillas con **exactamente la misma superficie por celda** (625 km²). Lo único que
cambia es dónde caen las líneas divisorias.

In [ ]:
def contar_en_grilla(puntos, celdas):
    """Cuenta puntos por celda, incluyendo las celdas vacías."""
    union = gpd.sjoin(puntos.to_crs(celdas.crs), celdas, predicate="within")
    conteo = union.groupby("celda_id").size()
    return conteo.reindex(celdas.celda_id, fill_value=0)


grillas = {
    "Cuadrados":              grilla(mendoza, 25, "cuadrado"),
    "Cuadrados, media celda": grilla(mendoza, 25, "cuadrado", desplazamiento=0.5),
    "Hexágonos, igual área":  grilla(mendoza, 25, "hexagono"),
}

filas = []
for nombre, celdas in grillas.items():
    conteo = contar_en_grilla(escuelas_mendoza, celdas)
    filas.append({
        "Grilla": nombre,
        "Celdas": len(celdas),
        "Área media (km²)": celdas.area_km2.mean(),
        "Media": conteo.mean(),
        "MÁXIMO": conteo.max(),
    })

pd.DataFrame(filas).round(1)

### ✅ Comprobación — ¿las tres grillas son realmente comparables?

In [ ]:
areas = [g.area_km2.mean() for g in grillas.values()]
print(f"Áreas medias por celda: {[round(a) for a in areas]} km²")
print(f"Diferencia máxima entre ellas: {100*(max(areas)-min(areas))/min(areas):.2f}%")
assert max(areas) / min(areas) < 1.01, "Las grillas no son comparables en superficie"
print("\n✓ Las tres grillas tienen la misma superficie por celda: la única diferencia es dónde cortan")

### 🔍 Interpretación

Las tres grillas tienen la misma superficie por celda, casi la misma cantidad de celdas y
**exactamente la misma media**. Pero el **máximo de escuelas en una celda** pasa de 283 a
167 con solo correr la grilla media celda: una caída del 41 %.

Ese máximo es lo que en un informe se llamaría *"la zona de mayor concentración educativa
de Mendoza"*. Y su valor depende de dónde dibujamos las líneas.

> La media es robusta a la zonificación; los **extremos no lo son**. Y en política pública
> los extremos son justamente lo que se mira: el barrio más carenciado, la zona más
> insegura, el área con menor cobertura.

### ▶️ Efecto de agregación

Ahora al revés: misma forma, distinto tamaño.

In [ ]:
filas = []
for lado_km in [10, 25, 50, 100]:
    celdas = grilla(mendoza, lado_km, "cuadrado")
    conteo = contar_en_grilla(escuelas_mendoza, celdas)
    densidad = conteo.values / celdas.area_km2.values * 100
    filas.append({
        "Lado (km)": lado_km,
        "Celdas": len(celdas),
        "Máx. escuelas/celda": conteo.max(),
        "Densidad máx. (/100 km²)": densidad.max(),
        "Coef. de variación": densidad.std() / densidad.mean(),
    })

pd.DataFrame(filas).round(2)

### 🔍 Interpretación

Al agrandar las celdas pasan dos cosas opuestas y ambas importan:

- El **conteo máximo por celda sube** (de 149 a 515): celdas más grandes contienen más cosas.
- La **densidad máxima baja** (de 149 a 5,2 escuelas cada 100 km²) y el **coeficiente de
  variación cae** (de 8,4 a 3,1): al agregar, promediamos zonas densas con zonas vacías y
  la variabilidad se suaviza.

Esto último es lo grave: **agregar oculta desigualdad**. Un mapa por provincia de un
indicador social siempre se verá más homogéneo que el mismo indicador por radio censal,
aunque la desigualdad subyacente sea idéntica. La elección de la escala determina cuánta
desigualdad es visible.

> ⚠️ **No hay una escala "correcta".** Hay una escala **adecuada al fenómeno**. Si estudiás
> accesibilidad a una escuela primaria, la unidad razonable es el radio censal o el barrio,
> porque nadie recorre 100 km para ir a primer grado. Si estudiás política educativa
> provincial, la provincia tiene sentido porque es la unidad donde se decide. Lo que no se
> puede es elegir la unidad por disponibilidad de datos y después no declararlo.

In [ ]:
fig, ejes = plt.subplots(1, 3, figsize=(14, 6))
for eje, (nombre, celdas) in zip(ejes, grillas.items()):
    celdas = celdas.copy()
    celdas["escuelas"] = contar_en_grilla(escuelas_mendoza, celdas).values
    celdas.plot(column="escuelas", cmap="OrRd", ax=eje,
                edgecolor="grey", linewidth=0.2, legend=True,
                legend_kwds={"shrink": 0.5})
    mendoza.to_crs(celdas.crs).boundary.plot(ax=eje, color="black", linewidth=0.8)
    eje.set_title(f"{nombre}\nmáx = {celdas.escuelas.max()}")
    eje.set_axis_off()
plt.tight_layout()
plt.show()

---

## 9. Efecto de borde

📽️ *Presentación Clase 2, diapositiva 15.*

### 🧭 Concepto

Todo análisis espacial se hace sobre una **zona de estudio** con un límite. Y ese límite
casi nunca es un límite del fenómeno: es un límite de nuestros datos.

Las unidades que caen cerca del borde tienen vecinos **del otro lado** que no estamos
contando. Cualquier medida que dependa del entorno —densidad, cantidad de vecinos,
promedio de la zona— queda **subestimada** ahí, y esa subestimación no es un ruido
aleatorio: es un sesgo sistemático que apunta siempre en la misma dirección.

Lo vamos a medir sobre una ventana de 80 × 80 km centrada en el Obelisco, comparando dos
formas de contar los vecinos de cada escuela: usando solo las escuelas de la ventana, o
usando todas las del país.

In [ ]:
from shapely.geometry import box
from scipy.spatial import cKDTree

escuelas_m = escuelas.to_crs(EQUIVALENTE)

# Ventana de estudio: 80 x 80 km centrada en el corazón del AMBA.
# Tomamos el centro de las escuelas de CABA sobre la capa YA proyectada:
# reproyectar un único punto suelto dispara una advertencia interna de pyproj.
caba = escuelas_m[escuelas_m.provincia == "Ciudad Autónoma de Buenos Aires"]
centro_x, centro_y = caba.geometry.x.mean(), caba.geometry.y.mean()

LADO = 40_000
ventana = box(centro_x - LADO, centro_y - LADO,
              centro_x + LADO, centro_y + LADO)

dentro = escuelas_m[escuelas_m.within(ventana)]
print(f"Escuelas dentro de la ventana: {len(dentro)}")
print(f"Escuelas en todo el país:      {len(escuelas_m)}")

In [ ]:
RADIO = 5_000  # contamos vecinos a 5 km

coord_dentro = np.array([(p.x, p.y) for p in dentro.geometry])
coord_todas  = np.array([(p.x, p.y) for p in escuelas_m.geometry])

# Vecinos contando SOLO la ventana, y contando TODO el país
vecinos_ventana = np.array([len(v) for v in
                            cKDTree(coord_dentro).query_ball_point(coord_dentro, RADIO)]) - 1
vecinos_todos   = np.array([len(v) for v in
                            cKDTree(coord_todas).query_ball_point(coord_dentro, RADIO)]) - 1

# ¿Qué escuelas están en la franja afectada por el borde?
borde = gpd.GeoSeries([ventana], crs=EQUIVALENTE).boundary.iloc[0]
en_franja = dentro.geometry.distance(borde).values < RADIO

print(f"Escuelas a menos de {RADIO/1000:.0f} km del borde: {en_franja.sum()} "
      f"({100*en_franja.mean():.0f}% de la muestra)")

### ✅ Comprobación

In [ ]:
resultado = pd.DataFrame({
    "Zona": ["Interior", "Franja de borde"],
    "Escuelas": [(~en_franja).sum(), en_franja.sum()],
    "Vecinos (solo ventana)": [vecinos_ventana[~en_franja].mean(),
                               vecinos_ventana[en_franja].mean()],
    "Vecinos (todo el país)": [vecinos_todos[~en_franja].mean(),
                               vecinos_todos[en_franja].mean()],
})
resultado["Subestimación (%)"] = (
    100 * (1 - resultado["Vecinos (solo ventana)"] / resultado["Vecinos (todo el país)"])
)
resultado.round(1)

### 🔍 Interpretación

En el **interior la subestimación es exactamente 0 %**, y en la **franja de borde es del
9,3 %**. Que el interior dé cero es la prueba de que el mecanismo es la truncación de los
datos y no otra cosa: no hay nada raro en el centro de la ventana.

Lo verdaderamente peligroso no es la magnitud del sesgo por unidad, sino **cuántas
unidades quedan afectadas**, que crece rápido con el radio de análisis:

| Radio | Escuelas en la franja | % de la muestra |
|---|---|---|
| 2 km | 99 | 3 % |
| 5 km | 318 | 9 % |
| 10 km | 704 | 19 % |
| 20 km | 1.591 | **43 %** |

Con un radio de 20 km, casi la mitad de las observaciones está contaminada.

> ⚠️ **Qué hacer.** Tres estrategias, en orden de preferencia: (1) traer datos de una zona
> más amplia que la de análisis y usar ese margen solo para el cálculo; (2) restringir las
> conclusiones al interior; (3) si no se puede ninguna de las dos, **declarar el problema**
> y no interpretar los valores del borde. Lo que nunca corresponde es presentar el borde
> como si fuera comparable con el interior.

---

## 10. Localización representada

📽️ *Presentación Clase 2, diapositiva 16.*

### 🧭 Concepto

Cuando un dato está agregado por área, tarde o temprano necesitamos **un punto** que la
represente: para calcular distancias, para dibujar un símbolo proporcional, para medir
vecindad. La elección habitual es el **centroide geométrico**, el centro de masa del
polígono suponiendo densidad uniforme.

Ese supuesto —densidad uniforme— casi nunca se cumple en variables sociales. La gente no
está repartida de manera pareja dentro de una provincia.

Comparamos el centroide geométrico de cada provincia contra el **centro de sus escuelas**,
que funciona como una aproximación a dónde está realmente la población.

In [ ]:
filas = []
for nombre in ["Buenos Aires", "Mendoza", "Santa Cruz", "Chubut", "Salta"]:
    poligono = provincias[provincias.provincia == nombre].to_crs(EQUIVALENTE)
    puntos   = escuelas[escuelas.provincia == nombre].to_crs(EQUIVALENTE)

    centro_geometrico = poligono.geometry.centroid.iloc[0]
    centro_escuelas   = gpd.points_from_xy(
        [puntos.geometry.x.mean()], [puntos.geometry.y.mean()]
    )[0]

    filas.append({
        "Provincia": nombre,
        "Escuelas": len(puntos),
        "Distancia entre centros (km)": centro_geometrico.distance(centro_escuelas) / 1000,
    })

pd.DataFrame(filas).round(1)

### 🔍 Interpretación

En Buenos Aires los dos puntos están a **186 km** de distancia. El centroide geométrico de
la provincia cae en el campo, cerca de Bolívar; el centro de sus escuelas cae hacia el
conurbano, que es donde vive la gente.

Si usáramos el centroide geométrico para calcular, por ejemplo, "la distancia media de un
bonaerense a un hospital de alta complejidad", el resultado estaría equivocado por cientos
de kilómetros, y equivocado **en contra** de la población más numerosa.

> ⚠️ **El error es más grave cuanto más desigual es el poblamiento y más grande la unidad.**
> Por eso las alternativas al centroide geométrico —el centroide ponderado por población,
> o un punto interior representativo— importan tanto en Argentina, donde la concentración
> urbana es extrema. Volvemos sobre esto en las clases 6 y 8.

---

## 11. La falacia ecológica

### 🧭 Concepto

Es el error de **atribuir a los individuos lo que se observó en los agregados**. Lo
describió Robinson en 1950, y sigue siendo uno de los errores más frecuentes en el uso de
datos territoriales en ciencias sociales.

El ejemplo clásico de Robinson: en los censos de Estados Unidos, los estados con mayor
proporción de población nacida en el extranjero tenían mayor alfabetización. Conclusión
tentadora: los inmigrantes eran más alfabetizados. Conclusión real: los inmigrantes se
asentaban en los estados **más ricos**, que ya tenían alta alfabetización. A nivel
individual la relación era la **inversa**.

La correlación entre agregados y la correlación entre individuos **son dos cantidades
distintas**, y no hay ninguna garantía de que se parezcan.

Lo comprobamos con nuestros datos: relación entre ser una escuela privada y el tamaño de
la matrícula, calculada primero escuela por escuela y después provincia por provincia.

In [ ]:
datos = escuelas.dropna(subset=["matricula", "sector"]).copy()
datos["es_privada"] = (datos["sector"] == "Privado").astype(int)

# Nivel individual: una fila por escuela
r_escuela = np.corrcoef(datos["es_privada"], datos["matricula"])[0, 1]

# Nivel agregado: una fila por provincia
por_provincia = datos.groupby("provincia").agg(
    proporcion_privadas=("es_privada", "mean"),
    matricula_media=("matricula", "mean"),
)
r_provincia = np.corrcoef(por_provincia["proporcion_privadas"],
                          por_provincia["matricula_media"])[0, 1]

print(f"Correlación a nivel ESCUELA   (n = {len(datos):,}):  r = {r_escuela:+.3f}")
print(f"Correlación a nivel PROVINCIA (n = {len(por_provincia)}):      r = {r_provincia:+.3f}")
print(f"\nLa correlación agregada es {r_provincia/r_escuela:.1f} veces más fuerte")

### ✅ Comprobación — miremos las medias, que no engañan

In [ ]:
print("Matrícula media por sector, a nivel escuela:")
print(datos.groupby("sector")["matricula"].agg(["count", "mean"]).round(0))

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(12, 5))

ejes[0].scatter(datos["es_privada"] + np.random.default_rng(0).normal(0, 0.04, len(datos)),
                datos["matricula"], s=2, alpha=0.1, color="#7570b3")
ejes[0].set_xticks([0, 1]); ejes[0].set_xticklabels(["Estatal", "Privada"])
ejes[0].set_ylabel("Matrícula"); ejes[0].set_ylim(0, 1500)
ejes[0].set_title(f"Nivel escuela (n = {len(datos):,})\nr = {r_escuela:+.3f}")

ejes[1].scatter(por_provincia["proporcion_privadas"] * 100,
                por_provincia["matricula_media"], s=60, color="#d95f02")
ejes[1].set_xlabel("% de escuelas privadas en la provincia")
ejes[1].set_ylabel("Matrícula media provincial")
ejes[1].set_title(f"Nivel provincia (n = {len(por_provincia)})\nr = {r_provincia:+.3f}")

plt.tight_layout()
plt.show()

### 🔍 Interpretación

A nivel escuela la relación es **débil** (r = +0,21): hay escuelas privadas chicas y
estatales enormes, y la nube de puntos es un manchón. A nivel provincia la relación se ve
**fuerte y limpia** (r = +0,76), 3,6 veces mayor.

El gráfico de la derecha es el que uno publicaría. Y es el que induce al error.

Si a partir de él alguien concluyera *"las escuelas privadas tienen más alumnos"*, estaría
diciendo algo que **estos datos apenas sostienen** a nivel individual. Lo que el gráfico
derecho muestra es una propiedad de las **provincias**: aquellas con más urbanización
tienen a la vez más oferta privada y escuelas más grandes, porque ambas cosas dependen de
la densidad de población.

> ⚠️ **La regla práctica.** Un dato agregado por área permite afirmar cosas sobre **áreas**,
> no sobre las personas que viven en ellas. "Los departamentos con más desocupación tienen
> más delito" no es lo mismo que "los desocupados delinquen más", y lo primero no es
> evidencia de lo segundo.
>
> La operación inversa —atribuir al conjunto lo observado en individuos— se llama **falacia
> atomista**, y también existe.

---

## 12. 🧪 Actividad integradora — ¿dónde está la zona de mayor concentración?

### La consigna

Vimos que el **máximo** de un indicador es lo que más se mueve al cambiar la grilla. Ahora
lo llevás a una pregunta de política pública: *¿dónde está el foco de concentración
escolar de una provincia?*

Sobre **una provincia a elección**:

1. Construí **tres grillas**: 25 km, 50 km y 25 km desplazada media celda.
2. Encontrá en cada una la celda con **más escuelas** (el "foco").
3. Medí **a qué distancia están esos focos entre sí**.
4. Escribí dos oraciones que un funcionario podría usar para justificar dónde poner un
   supervisor escolar, y decí cuál de las tres grillas sostiene cada una.

In [ ]:
# Cambiá la provincia y ejecutá
PROVINCIA = "Mendoza"     # probá también con "Salta", "Chaco", "Río Negro"

zona   = provincias[provincias.provincia == PROVINCIA]
puntos = escuelas[escuelas.provincia == PROVINCIA]

configuraciones = {
    "25 km":            grilla(zona, 25, "cuadrado"),
    "50 km":            grilla(zona, 50, "cuadrado"),
    "25 km desplazada": grilla(zona, 25, "cuadrado", desplazamiento=0.5),
}

focos = {}
for nombre, celdas in configuraciones.items():
    # ⚠️ grilla() devuelve en el CRS de la zona, que es geográfico (grados).
    # Para medir distancias entre focos HAY que proyectar primero.
    celdas = celdas.to_crs(EQUIVALENTE).copy()
    celdas["escuelas"] = contar_en_grilla(puntos, celdas).values

    foco = celdas.nlargest(1, "escuelas").iloc[0]
    focos[nombre] = foco.geometry.centroid
    print(f"{nombre:20} foco con {foco['escuelas']:.0f} escuelas")

### ✅ Comprobación — el error que casi cometemos

Esta comprobación no está de adorno. Al preparar esta clase medimos las distancias **sin
proyectar**, y todas dieron `0.0 km`. No era que los focos coincidieran: `.distance()`
devolvía **grados**, y dividir 0,16 grados por 1.000 da 0,00016, que se imprime como cero.

Un resultado de "0 km" es perfectamente creíble —significaría que la grilla no importa— y
habría invertido la conclusión de toda la clase. Por eso se comprueba **antes** de
interpretar.

In [ ]:
from sig_utils import chequear_crs

muestra = configuraciones["25 km"].to_crs(EQUIVALENTE)
chequear_crs(muestra, proyectado=True, nombre="grilla para medir distancias")

In [ ]:
nombres = list(focos)
print(f"Distancia entre los focos señalados por cada grilla:\n")
for i in range(len(nombres)):
    for j in range(i + 1, len(nombres)):
        distancia_km = focos[nombres[i]].distance(focos[nombres[j]]) / 1000
        print(f"  {nombres[i]:20} vs {nombres[j]:20} {distancia_km:6.1f} km")

### Tu respuesta

**Máximo de escuelas según cada grilla:** ▸ 25 km: ___ · 50 km: ___ · desplazada: ___

**Distancia máxima entre los focos:** ▸ ___ km

**Las dos oraciones para el funcionario:**

1. ▸
2. ▸

**Qué grilla usarías y por qué (dos renglones):** ▸

> 💡 No hay una respuesta correcta única. Se evalúa que la justificación conecte la escala
> elegida con **el fenómeno**: a qué distancia razonable se supervisa una escuela, y qué
> unidad usa efectivamente quien toma la decisión. En Mendoza los focos quedan a entre
> 17 y 25 km entre sí: suficiente para que el supervisor viva en otra ciudad.

---

## 13. 🤖 Actividad con IA generativa — el MAUP no es deducible

### Consigna

1. Pedile a un asistente, **sin darle los datos**:

   > Tengo las 874 escuelas primarias de Mendoza y una grilla de celdas cuadradas de
   > 25 km de lado. Si desplazo la grilla media celda hacia el nordeste, manteniendo el
   > mismo tamaño, ¿cambiaría mucho la cantidad máxima de escuelas en una celda? Dame un
   > porcentaje aproximado.

2. Anotá el porcentaje que te dio.
3. Ejecutá la celda de verificación.
4. Completá el veredicto.

### ¿Por qué esta actividad?

El efecto de zonificación **no se puede deducir**: depende de cómo estén distribuidos los
puntos concretos, y solo se sabe midiéndolo. Un asistente va a dar igual una respuesta
—probablemente "un cambio pequeño, del 5 al 10 %"— porque es lo que suena razonable.

El valor real que medimos hoy fue del **41 %**.

In [ ]:
# Pegá acá lo que respondió el asistente
respuesta_ia = """
(pegar acá)
"""

# Verificación
sin_desplazar = contar_en_grilla(escuelas_mendoza, grilla(mendoza, 25, "cuadrado")).max()
desplazada    = contar_en_grilla(escuelas_mendoza,
                                 grilla(mendoza, 25, "cuadrado", desplazamiento=0.5)).max()

print(f"Máximo sin desplazar: {sin_desplazar}")
print(f"Máximo desplazada:    {desplazada}")
print(f"Cambio real:          {100*abs(desplazada-sin_desplazar)/sin_desplazar:.0f}%")

### Veredicto

| | |
|---|---|
| Porcentaje estimado por el asistente | ▸ |
| Porcentaje real | ▸ |
| ¿Aclaró que dependía de los datos concretos? | ▸ sí / no |
| ¿Ofreció una forma de comprobarlo? | ▸ sí / no |

**En una frase, ¿qué distingue una pregunta que un asistente puede responder de una que
solo se responde midiendo?** ▸

---

## 14. Cierre

### Glosario de la clase

| Término | Definición |
|---|---|
| **Primera ley de la geografía** | Las cosas cercanas se parecen más entre sí que las lejanas (Tobler, 1970). |
| **Autocorrelación espacial** | Que el valor de una variable en un lugar dependa de su valor en los lugares vecinos. Viola el supuesto de independencia. |
| **Escala de análisis** | Nivel de detalle de la medición. Determina qué se puede observar y cuánto mide lo que se mide. |
| **Generalización** | Simplificación de una geometría reduciendo vértices. |
| **MAUP** | Que los resultados dependan de las unidades areales elegidas, que son arbitrarias. |
| **Efecto de agregación** | Componente del MAUP asociado al **tamaño** de las unidades. |
| **Efecto de zonificación** | Componente del MAUP asociado a la **forma o posición** de las unidades. |
| **Efecto de borde** | Sesgo en los cálculos de las unidades cercanas al límite de la zona de estudio. |
| **Localización representada** | El punto que se elige para representar un área; el centroide geométrico supone densidad uniforme. |
| **Falacia ecológica** | Atribuir a los individuos lo observado en datos agregados por área. |

### Autoevaluación

1. Tenés dos mapas de pobreza del mismo país: uno por provincia y otro por radio censal.
   El primero se ve más homogéneo. ¿Es una conclusión sobre el país o sobre los mapas?
2. Un colega calculó la densidad de comercios en un barrio y encontró que las manzanas del
   borde tienen la mitad. ¿Qué le preguntarías antes de creerle?
3. *"Los departamentos con mayor porcentaje de población migrante tienen mayor tasa de
   empleo."* ¿Qué afirmación **no** se sigue de esto?

### 📦 Material opcional

`Clase_2_opcional.ipynb`: desagregación dasimétrica como anticipo de la Clase 8, y
zonificación con unidades irregulares reales (departamentos vs. radios censales).

### Tarea para la Clase 3

Retomá la pregunta territorial que escribiste para hoy y agregale un párrafo que responda:

1. **Qué unidad territorial** vas a usar y por qué **esa** y no una más chica o más grande.
2. **Qué límite** tiene tu zona de estudio y si hay efecto de borde. Si lo hay, cómo pensás
   tratarlo.
3. **Qué afirmación NO vas a poder hacer** con datos agregados a esa unidad.

Media carilla. El punto 3 es el importante.

---

### Y lo que quedó pendiente

Hoy usamos dos sistemas de coordenadas —uno para áreas y otro para distancias— sin
explicar por qué. También dijimos que medir en grados no sirve, sin justificarlo del todo.
**La Clase 3 se ocupa exactamente de eso**, y vas a ver que la diferencia entre elegir bien
y elegir mal puede ser del 55 % en la superficie de Argentina.